# Active Development — BeerQA Dev20 Standard RAG

It contains:
- BeerQA dev loading
- Dev20 targeted Wikipedia corpus
- BM25 + DPR/FAISS
- RRF hybrid retrieval for Standard RAG
- DeepSeek answer generation
- Exact Match / F1 evaluation
- Title Recall@10 / Title Precision@10 development metrics

Expensive artifacts are loaded from disk when they already exist.

## 1. Environment and Paths

Install packages once if missing:

`pip install rank-bm25 transformers torch faiss-cpu sentencepiece openai`

In [38]:
from pathlib import Path
from collections import Counter
import os
import json
import bz2
import re
import string
import getpass
import numpy as np
import torch
import faiss

from rank_bm25 import BM25Okapi
from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
)
from openai import OpenAI

WIKI_ROOT = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\enwiki-20200801-pages-articles-tokenized\enwiki-20200801-pages-articles-tokenized"
)

BEERQA_DEV_PATH = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\beerqa_dev_v1.0.json"
)

OUTPUT_DIR = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\processed_data"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEERQA_TARGET_ARTICLES_PATH = OUTPUT_DIR / "beerqa_dev20_target_articles.jsonl"
BEERQA_CORPUS_PATH = OUTPUT_DIR / "beerqa_dev20_wikipedia_corpus.jsonl"
BEERQA_DPR_EMBEDDINGS_PATH = OUTPUT_DIR / "beerqa_dev20_dpr_embeddings.npy"
BEERQA_FAISS_PATH = OUTPUT_DIR / "beerqa_dev20.faiss"
STANDARD_RAG_DEV20_RESULTS_PATH = OUTPUT_DIR / "standard_rag_dev20_results.jsonl"
STANDARD_RAG_DEV20_EVAL_PATH = OUTPUT_DIR / "standard_rag_dev20_evaluated.jsonl"

print("BeerQA dev exists:", BEERQA_DEV_PATH.exists())
print("Processed-data folder:", OUTPUT_DIR)

BeerQA dev exists: True
Processed-data folder: C:\Users\Wei Khang\Documents\Master\FYP\processed_data


## 2. Shared Text Helpers

In [ ]:
# Converts the input into a string.
# Replaces underscores (_) with spaces.
# Removes unnecessary spaces at the beginning and end.
# Converts uppercase letters to lowercase.
# Removes extra spaces between words.

def normalize_title(title):
    return " ".join(
        str(title)
        .replace("_", " ")
        .strip()
        .lower()
        .split()
    )


# Extracts the article title and text
# Checks the data type whether the item is a string
# Removes unnecessary whitespace and unwanted sentences
# Stores and returns cleaned sentences

def clean_article_text(article):
    title = article.get("title", "").strip() 
    raw_text = article.get("text", []) 

    cleaned_sentences = []

    for item in raw_text:
        if not isinstance(item, str):
            continue

        sentence = " ".join(item.split())

        if not sentence:
            continue

        if sentence.lower() == title.lower():
            continue

        cleaned_sentences.append(sentence)

    return cleaned_sentences


# Split cleaned text into 100-word chunks with 20-word overlap while preserving article metadata.
def create_chunks(
    sentences,
    article_id,
    title,
    url,
    max_words=100,
    overlap_words=20 # Reduce the risk of losing important information when splitting Wikipedia articles into smaller chunks
):
    chunks = []
    current_words = []

    for sentence in sentences:
        words = sentence.split()

        # Calc the remaining space for current chunk
        # Check whether all remaining words can fit
        # Selects only the words that fit into the space
        # Keep the last 20 words of the completed chunk to reuse in the next chunk
        while words:
            remaining_space = max_words - len(current_words) 

            if len(words) <= remaining_space: 
                current_words.extend(words)
                words = []
            else:
                current_words.extend(words[:remaining_space]) 

                chunks.append({
                    "passage_id": f"{article_id}_{len(chunks)}",
                    "article_id": article_id,
                    "title": title,
                    "url": url,
                    "text": " ".join(current_words),
                })

                overlap = current_words[-overlap_words:] if overlap_words > 0 else [] 
                current_words = overlap
                words = words[remaining_space:]

    if current_words:
        chunks.append({
            "passage_id": f"{article_id}_{len(chunks)}",
            "article_id": article_id,
            "title": title,
            "url": url,
            "text": " ".join(current_words),
        })

    return chunks


def article_to_chunks(article):
    cleaned_sentences = clean_article_text(article)

    if not cleaned_sentences:
        return []

    return create_chunks(
        sentences=cleaned_sentences,
        article_id=article.get("id"),
        title=article.get("title", ""),
        url=article.get("url", ""),
        max_words=100,
        overlap_words=20,
    )


def bm25_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

## 3. Load BeerQA Dev and Select the 20-Question Development Sample

In [40]:
with open(BEERQA_DEV_PATH, "r", encoding="utf-8") as f:
    beerqa_dev = json.load(f)

beerqa_records = beerqa_dev["data"]

print("Total BeerQA dev records:", len(beerqa_records))
print("Top-level keys:", beerqa_dev.keys())

Total BeerQA dev records: 14121
Top-level keys: dict_keys(['version', 'split', 'data'])


In [ ]:
# Extract the article titles and collect unique titles
def get_context_titles(record):
    titles = []

    for item in record.get("context", []):
        if isinstance(item, list) and len(item) >= 1:
            title = item[0]

            if title:
                titles.append(normalize_title(title))

    return list(dict.fromkeys(titles))


DEV_SAMPLE_SIZE = 20
beerqa_sample = beerqa_records[:DEV_SAMPLE_SIZE]

target_titles = set()

for record in beerqa_sample:
    for title in get_context_titles(record):
        target_titles.add(title)

print("Development questions:", len(beerqa_sample))
print("Unique target Wikipedia titles:", len(target_titles))

Development questions: 20
Unique target Wikipedia titles: 29


## 4. Load or Build the BeerQA Dev20 Wikipedia Corpus

Priority:
1. Load the processed `beerqa_dev20_wikipedia_corpus.jsonl` if it exists.
2. Otherwise load saved target articles and re-chunk them.
3. Only scan all Wikipedia `.bz2` files if neither saved file exists.

In [ ]:
if BEERQA_CORPUS_PATH.exists():
    print("Processed BeerQA corpus already exists. Loading...")

    beerqa_corpus = []

    with open(BEERQA_CORPUS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            beerqa_corpus.append(json.loads(line))

elif BEERQA_TARGET_ARTICLES_PATH.exists():
    print("Raw target articles exist. Loading and chunking...")

    target_articles = {}

    with open(BEERQA_TARGET_ARTICLES_PATH, "r", encoding="utf-8") as f:
        for line in f:
            article = json.loads(line)
            target_articles[normalize_title(article.get("title", ""))] = article

    beerqa_corpus = []

    for article in target_articles.values():
        beerqa_corpus.extend(article_to_chunks(article))

    with open(BEERQA_CORPUS_PATH, "w", encoding="utf-8") as f:
        for chunk in beerqa_corpus:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n") # convert into json format and save into the file

    print("Saved processed corpus:", BEERQA_CORPUS_PATH)

else:
    print("No saved Dev20 corpus found. Scanning Wikipedia once...")

    bz2_files = sorted(WIKI_ROOT.rglob("*.bz2"))

    if not bz2_files:
        raise FileNotFoundError("No Wikipedia .bz2 files found under WIKI_ROOT.")

    normalized_targets = {normalize_title(title) for title in target_titles}
    target_articles = {}

    for file_num, file_path in enumerate(bz2_files, start=1):
        try:
            with bz2.open(
                file_path,
                "rt", # read text
                encoding="utf-8",
                errors="replace",
            ) as f:

                for line in f:
                    try:
                        article = json.loads(line)
                    except json.JSONDecodeError:
                        continue

                    article_title = normalize_title(
                        article.get("title", "")
                    )

                    if (
                        article_title in normalized_targets
                        and article_title not in target_articles
                    ):
                        target_articles[article_title] = article

                        print(
                            f"Found {len(target_articles)} / "
                            f"{len(normalized_targets)}:",
                            article.get("title"),
                        ) # Print search progress

        except Exception as e:
            print("Problem reading:", file_path, "|", e)
            continue

        if file_num % 500 == 0:
            print(f"Scanned {file_num} / {len(bz2_files)} files")

        if len(target_articles) == len(normalized_targets):
            print("All target articles found.")
            break

    missing_titles = normalized_targets - set(target_articles.keys())

    print("Requested titles:", len(normalized_targets))
    print("Found titles:", len(target_articles))
    print("Missing titles:", len(missing_titles))

    if missing_titles:
        print("Missing Wikipedia titles:")
        for title in sorted(missing_titles):
            print("-", title)

        raise RuntimeError("Some target Wikipedia articles were not found.")

    with open(BEERQA_TARGET_ARTICLES_PATH, "w", encoding="utf-8") as f:
        for article in target_articles.values():
            f.write(json.dumps(article, ensure_ascii=False) + "\n")

    beerqa_corpus = []

    for article in target_articles.values():
        beerqa_corpus.extend(article_to_chunks(article))

    with open(BEERQA_CORPUS_PATH, "w", encoding="utf-8") as f:
        for chunk in beerqa_corpus:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    print("Saved target articles:", BEERQA_TARGET_ARTICLES_PATH)
    print("Saved processed corpus:", BEERQA_CORPUS_PATH)

print("BeerQA development passages:", len(beerqa_corpus))

Processed BeerQA corpus already exists. Loading...
BeerQA development passages: 1674


In [43]:
passage_ids = [chunk["passage_id"] for chunk in beerqa_corpus]
word_counts = [len(chunk["text"].split()) for chunk in beerqa_corpus]
titles = {normalize_title(chunk["title"]) for chunk in beerqa_corpus}

print("Total passages:", len(beerqa_corpus))
print("Unique passage IDs:", len(set(passage_ids)))
print("Unique Wikipedia titles:", len(titles))
print("Maximum chunk words:", max(word_counts))

assert len(passage_ids) == len(set(passage_ids))
assert max(word_counts) <= 100

print("PASS: BeerQA corpus validation successful.")

Total passages: 1674
Unique passage IDs: 1674
Unique Wikipedia titles: 29
Maximum chunk words: 100
PASS: BeerQA corpus validation successful.


## 5. BM25 Index and Search
1. Convert the question into lowercase words.
2. Calculate BM25 relevance scores for the indexed passages.



3. Sort the passages according to their scores.

4. Return the top 10 results.

In [ ]:
beerqa_bm25_documents = [
    chunk["title"] + " " + chunk["text"]
    for chunk in beerqa_corpus
]

beerqa_tokenized_corpus = [
    bm25_tokenize(document)
    for document in beerqa_bm25_documents
]

# BM25 uses these statistics to calculate relevance scores:
# 1. Term frequency (TF) — How frequently a word appears in a passage.
# 2. Inverse document frequency (IDF) — How informative a word is based on how commonly it appears across the corpus.
# 3. Document length — How many tokens each passage contains.
# 4. Average document length — The average number of tokens across all passages.
beerqa_bm25 = BM25Okapi(beerqa_tokenized_corpus)

print("BeerQA BM25 index created:", len(beerqa_tokenized_corpus), "passages")

BeerQA BM25 index created: 1674 passages


In [ ]:
def beerqa_bm25_search(query, top_k=10):
    tokenized_query = bm25_tokenize(query)
    scores = beerqa_bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k] # Sort and select the highest-scoring passages

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = beerqa_corpus[idx]

        results.append({
            "rank": rank,
            "passage_id": chunk["passage_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "bm25_score": float(scores[idx]),
        })

    return results

## 6. DPR Models, Cached Embeddings, and FAISS

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

ctx_model_name = "facebook/dpr-ctx_encoder-multiset-base"
question_model_name = "facebook/dpr-question_encoder-multiset-base"

# Converts Wikipedia passages into numerical vectors (numerical embeddings)
ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained(ctx_model_name) 
ctx_encoder = DPRContextEncoder.from_pretrained(ctx_model_name).to(device) 

# Converts the input question into a numerical vector (numerical embeddings)
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(question_model_name) 
question_encoder = DPRQuestionEncoder.from_pretrained(question_model_name).to(device)

ctx_encoder.eval()
question_encoder.eval()

print("DPR models loaded.")

Device: cpu


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 32752.41it/s]
[transformers] DPRContextEncoder LOAD REPORT from: facebook/dpr-ctx_encoder-multiset-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
ctx_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
ctx_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 48869.05it/s]
[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-multiset-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored 

DPR models loaded.


In [ ]:
# Process passages in batches and converts them into 768-dimensional numerical vectors
def encode_passages(documents, tokenizer, model, batch_size=32):
    embeddings = []

    for start in range(0, len(documents), batch_size):
        batch_texts = documents[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True, # padding tokens to shorter passages so all tokenized sequences in the batch have the same length.
            truncation=True, # Cuts off tokens if a passage exceeds the allowed length
            max_length=256, #256 tokens
            return_tensors="pt",
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs) # generate the embedding

            batch_embeddings = (
                outputs.pooler_output
                .cpu()
                .numpy()
                .astype("float32")
            )

        embeddings.append(batch_embeddings)

        print(
            f"Encoded {min(start + batch_size, len(documents))} / "
            f"{len(documents)} passages"
        )

    return np.vstack(embeddings)

In [48]:
beerqa_dpr_documents = [
    chunk["title"] + " " + chunk["text"]
    for chunk in beerqa_corpus
]

regenerate_embeddings = True

if BEERQA_DPR_EMBEDDINGS_PATH.exists():
    print("Saved DPR embeddings found. Loading...")

    beerqa_passage_embeddings = np.load(BEERQA_DPR_EMBEDDINGS_PATH)

    if (
        beerqa_passage_embeddings.ndim == 2
        and beerqa_passage_embeddings.shape[0] == len(beerqa_corpus)
        and beerqa_passage_embeddings.shape[1] == 768
    ):
        regenerate_embeddings = False
        print("Loaded embedding shape:", beerqa_passage_embeddings.shape)
    else:
        print("Saved embeddings do not match the current corpus.")

if regenerate_embeddings:
    print("Generating DPR embeddings...")

    beerqa_passage_embeddings = encode_passages(
        documents=beerqa_dpr_documents,
        tokenizer=ctx_tokenizer,
        model=ctx_encoder,
        batch_size=32,
    )

    np.save(
        BEERQA_DPR_EMBEDDINGS_PATH,
        beerqa_passage_embeddings,
    )

    print("Saved DPR embeddings:", BEERQA_DPR_EMBEDDINGS_PATH)

assert beerqa_passage_embeddings.shape == (len(beerqa_corpus), 768)
print("PASS: DPR embeddings match BeerQA corpus.")

Saved DPR embeddings found. Loading...
Loaded embedding shape: (1674, 768)
PASS: DPR embeddings match BeerQA corpus.


In [ ]:
rebuild_faiss = True

if BEERQA_FAISS_PATH.exists():
    print("Saved FAISS index found. Loading...")

    beerqa_faiss_index = faiss.read_index(str(BEERQA_FAISS_PATH))

    if (
        beerqa_faiss_index.ntotal == len(beerqa_corpus)
        and beerqa_faiss_index.d == beerqa_passage_embeddings.shape[1]
    ):
        rebuild_faiss = False
        print("Loaded FAISS vectors:", beerqa_faiss_index.ntotal)
    else:
        print("Saved FAISS index does not match the current corpus.")

if rebuild_faiss:
    print("Building FAISS index...")

    beerqa_faiss_index = faiss.IndexFlatIP(
        beerqa_passage_embeddings.shape[1]
    ) # creates a FAISS index that uses inner product (dot product) to measure vector similarity

    beerqa_faiss_index.add(
        beerqa_passage_embeddings.astype("float32")
    )

    faiss.write_index(
        beerqa_faiss_index,
        str(BEERQA_FAISS_PATH),
    )

    print("Saved FAISS index:", BEERQA_FAISS_PATH)

assert beerqa_faiss_index.ntotal == len(beerqa_corpus)
print("PASS: FAISS index matches BeerQA corpus.")

Saved FAISS index found. Loading...
Loaded FAISS vectors: 1674
PASS: FAISS index matches BeerQA corpus.


In [50]:
# Converts the question into an embedding, searches the FAISS index and returns passages with the highest vector similarity scores
def beerqa_dpr_search(query, top_k=10):
    inputs = question_tokenizer(
        query,
        return_tensors="pt",# "PyTorch"
        truncation=True,
        max_length=256,
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():# because no trained a dpr model so no gradient needed and i just want the output
        query_embedding = (
            question_encoder(**inputs)
            .pooler_output
            .cpu()
            .numpy()
            .astype("float32")
        )

    scores, indices = beerqa_faiss_index.search(
        query_embedding,
        top_k,
    )

    results = []

    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        chunk = beerqa_corpus[idx]

        results.append({
            "rank": rank,
            "passage_id": chunk["passage_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "dpr_score": float(score),
        })

    return results

## 7. Standard RAG Hybrid Retrieval with RRF
1. BM25 retrieves its top 10 passages.
2. DPR retrieves its top 10 passages.
3. Results are merged using passage IDs.
4. Duplicate passages are removed.
5. An RRF score is calculated from their retrieval ranks.
6. The highest-ranked 10 passages are returned.

In [ ]:
# Combines BM25 and DPR into one hybrid retrieval system
def beerqa_standard_hybrid_search(
    query,
    top_k_each=10,
    final_top_k=10,
    rrf_k=60
):
    bm25_results = beerqa_bm25_search(query, top_k=top_k_each)
    dpr_results = beerqa_dpr_search(query, top_k=top_k_each)

    combined = {}

    for rank, result in enumerate(bm25_results, start=1):
        pid = result["passage_id"]

        if pid not in combined:
            combined[pid] = {
                "passage_id": pid,
                "article_id": result["article_id"],
                "title": result["title"],
                "text": result["text"],
                "bm25_rank": None,
                "dpr_rank": None,
                "bm25_score": None,
                "dpr_score": None,
                "rrf_score": 0.0,
                "source": [],
            }

        combined[pid]["bm25_rank"] = rank
        combined[pid]["bm25_score"] = result["bm25_score"]
        combined[pid]["source"].append("BM25")
        combined[pid]["rrf_score"] += 1 / (rrf_k + rank)

    for rank, result in enumerate(dpr_results, start=1):
        pid = result["passage_id"]

        if pid not in combined:
            combined[pid] = {
                "passage_id": pid,
                "article_id": result["article_id"],
                "title": result["title"],
                "text": result["text"],
                "bm25_rank": None,
                "dpr_rank": None,
                "bm25_score": None,
                "dpr_score": None,
                "rrf_score": 0.0,
                "source": [],
            }

        combined[pid]["dpr_rank"] = rank
        combined[pid]["dpr_score"] = result["dpr_score"]

        if "DPR" not in combined[pid]["source"]:
            combined[pid]["source"].append("DPR")

        combined[pid]["rrf_score"] += 1 / (rrf_k + rank)

    ranked_results = sorted(
        combined.values(),
        key=lambda x: x["rrf_score"],
        reverse=True,
    )

    return ranked_results[:final_top_k]

## 8. DeepSeek Answer Generation

No API key is stored in this notebook. Enter it when prompted.

In [52]:
if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass(
        "Enter your DeepSeek API key: "
    )

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

DEEPSEEK_MODEL = "deepseek-v4-flash"
print("DeepSeek client ready.")

DeepSeek client ready.


In [53]:
# Prepare retrieved passages in a structured format so that DeepSeek can understand which Wikipedia information is available
def build_standard_context(retrieved_results, max_passages=10):
    context_blocks = []

    for i, result in enumerate(retrieved_results[:max_passages], start=1):
        context_blocks.append(
            f"[Passage {i}]\n"
            f"Title: {result['title']}\n"
            f"Text: {result['text']}"
        )

    return "\n\n".join(context_blocks)


In [54]:
# Send original question and retrieved evidence to DeepSeek
# The prompt instructs the model to produce a short factual answer using only the supplied context
def generate_standard_answer(question, context, model=DEEPSEEK_MODEL):
    system_prompt = """
You are an open-domain question answering system.

Answer the question using only the retrieved Wikipedia context provided.

Rules:
1. Use the retrieved context as the evidence for your answer.
2. Do not use unsupported outside information.
3. Return only the short factual answer.
4. Do not provide explanations unless the question requires them.
5. If the retrieved context does not contain enough evidence to answer the question, return:
INSUFFICIENT_EVIDENCE
""".strip()

    user_prompt = f"""
Question:
{question}

Retrieved Wikipedia Context:
{context}

Answer:
""".strip()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=100,
        temperature=0.0,
        extra_body={"thinking": {"type": "disabled"}},
    )

    return response.choices[0].message.content.strip()

In [55]:
def beerqa_standard_rag(
    question,
    top_k_each=10,
    final_top_k=10
):
    retrieved_results = beerqa_standard_hybrid_search(
        query=question,
        top_k_each=top_k_each,
        final_top_k=final_top_k,
    )

    context = build_standard_context(
        retrieved_results,
        max_passages=final_top_k,
    )

    generated_answer = generate_standard_answer(
        question=question,
        context=context,
    )

    return {
        "question": question,
        "generated_answer": generated_answer,
        "retrieved_passages": retrieved_results,
        "context": context,
    }

## 9. One-Question End-to-End Smoke Test

In [56]:
test_record = beerqa_sample[0]

print("Question:", test_record["question"])
print("Gold answer:", test_record["answers"])

beerqa_result = beerqa_standard_rag(test_record["question"])
print("Standard RAG answer:", beerqa_result["generated_answer"])

Question: Which genus contains more species, Ortegocactus or Eschscholzia?
Gold answer: ['Eschscholzia']


Standard RAG answer: Eschscholzia


In [58]:
for rank, passage in enumerate(
    beerqa_result["retrieved_passages"],
    start=1
):
    print(
        rank,
        "|",
        passage["passage_id"],
        "|",
        passage["title"],
        "| RRF:",
        round(passage["rrf_score"], 6)
    )

1 | 2678316_0 | Ortegocactus | RRF: 0.032787
2 | 250512_0 | Eschscholzia | RRF: 0.031754
3 | 250512_2 | Eschscholzia | RRF: 0.031754
4 | 250512_1 | Eschscholzia | RRF: 0.031746
5 | 25614_23 | Race (human categorization) | RRF: 0.015385
6 | 12551_50 | Gymnastics | RRF: 0.015385
7 | 25614_24 | Race (human categorization) | RRF: 0.015152
8 | 12551_9 | Gymnastics | RRF: 0.015152
9 | 42370_1 | History of Belgium | RRF: 0.014925
10 | 12551_41 | Gymnastics | RRF: 0.014925


## 10. Exact Match and F1 Evaluation

In [59]:
# Converts answers to a standardized format by removing differences in capitalization, punctuation, articles and extra spaces
def normalize_answer(text):
    text = str(text).lower()

    text = "".join(
        ch for ch in text
        if ch not in string.punctuation
    )

    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = " ".join(text.split())

    return text

# Returns 1 when the normalized predicted answer matches the reference answer exactly, otherwise 0
def exact_match_score(prediction, ground_truth):
    return int(
        normalize_answer(prediction)
        ==
        normalize_answer(ground_truth)
    )

# Measures how many words overlap between the predicted and reference answers using token-level precision and recall
def f1_score(prediction, ground_truth):
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()

    if not prediction_tokens or not ground_truth_tokens:
        return float(prediction_tokens == ground_truth_tokens)

    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(prediction_tokens)
    recall = num_same / len(ground_truth_tokens)

    return 2 * precision * recall / (precision + recall)

#Calculates EM and F1, taking the best score when multiple reference answers exist
def evaluate_answer(prediction, gold_answers):
    if not gold_answers:
        return {"em": 0, "f1": 0.0}

    return {
        "em": max(
            exact_match_score(prediction, gold)
            for gold in gold_answers
        ),
        "f1": max(
            f1_score(prediction, gold)
            for gold in gold_answers
        ),
    }

In [60]:
def evaluate_standard_rag_records(records, limit=None):
    if limit is not None:
        records = records[:limit]

    results = []

    for index, record in enumerate(records, start=1):
        print(f"\nRunning {index} / {len(records)}")
        print("Question:", record["question"])

        try:
            rag_result = beerqa_standard_rag(record["question"])
            prediction = rag_result["generated_answer"]

            metrics = evaluate_answer(
                prediction,
                record["answers"],
            )

            retrieved = []

            for passage in rag_result["retrieved_passages"]:
                retrieved.append({
                    "passage_id": passage["passage_id"],
                    "title": passage["title"],
                    "rrf_score": passage.get("rrf_score"),
                })

            result = {
                "id": record["id"],
                "src": record["src"],
                "question": record["question"],
                "gold_answers": record["answers"],
                "prediction": prediction,
                "em": metrics["em"],
                "f1": metrics["f1"],
                "retrieved_passages": retrieved,
                "error": None,
            }

            results.append(result)

            print("Gold:", record["answers"])
            print("Prediction:", prediction)
            print("EM:", metrics["em"])
            print("F1:", round(metrics["f1"], 4))

        except Exception as e:
            print("ERROR:", e)

            results.append({
                "id": record.get("id"),
                "src": record.get("src"),
                "question": record.get("question"),
                "gold_answers": record.get("answers"),
                "prediction": None,
                "em": None,
                "f1": None,
                "retrieved_passages": [],
                "error": str(e),
            })

    return results

## 11. Load Existing Dev20 Results or Run Them Intentionally

In [61]:
if STANDARD_RAG_DEV20_RESULTS_PATH.exists():
    standard_rag_test20 = []

    with open(
        STANDARD_RAG_DEV20_RESULTS_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            standard_rag_test20.append(json.loads(line))

    print("Loaded saved Standard RAG Dev20 results:", len(standard_rag_test20))

elif STANDARD_RAG_DEV20_EVAL_PATH.exists():
    standard_rag_test20 = []

    with open(
        STANDARD_RAG_DEV20_EVAL_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            standard_rag_test20.append(json.loads(line))

    print("Loaded saved evaluated Dev20 results:", len(standard_rag_test20))

else:
    standard_rag_test20 = []
    print(
        "No saved Dev20 results found. "
        "Run the next experiment cell intentionally to make API calls."
    )

Loaded saved Standard RAG Dev20 results: 20


In [62]:
# standard_rag_test20 = evaluate_standard_rag_records(
#     beerqa_sample,
#     limit=20,
# )
#
# with open(
#     STANDARD_RAG_DEV20_RESULTS_PATH,
#     "w",
#     encoding="utf-8",
# ) as f:
#     for result in standard_rag_test20:
#         f.write(json.dumps(result, ensure_ascii=False) + "\n")
#
# print("Saved:", STANDARD_RAG_DEV20_RESULTS_PATH)

In [63]:
if standard_rag_test20:
    successful_results = [
        result
        for result in standard_rag_test20
        if result.get("error") is None
    ]

    average_em = sum(
        result["em"]
        for result in successful_results
    ) / len(successful_results)

    average_f1 = sum(
        result["f1"]
        for result in successful_results
    ) / len(successful_results)

    print("Total questions:", len(standard_rag_test20))
    print("Successfully completed:", len(successful_results))
    print("Errors:", len(standard_rag_test20) - len(successful_results))
    print("Average EM:", round(average_em, 4))
    print("Average F1:", round(average_f1, 4))

Total questions: 20
Successfully completed: 20
Errors: 0
Average EM: 0.35
Average F1: 0.5487


## 12. Development Retrieval Metrics

These are **title/document-level** development metrics, not final passage-level BeerQA metrics.

In [64]:
# Extracts the annotated Wikipedia titles associated with the question
def get_gold_titles(record):
    gold_titles = []

    for item in record.get("context", []):
        if isinstance(item, list) and len(item) >= 1:
            title = normalize_title(item[0])

            if title and title not in gold_titles:
                gold_titles.append(title)

    return gold_titles

# Extracts the titles of the retrieved passages
def get_retrieved_titles(result, top_k=10):
    return [
        normalize_title(passage["title"])
        for passage in result["retrieved_passages"][:top_k]
    ]

# Calculates the proportion of expected Wikipedia titles found among retrieved results
def title_recall_at_k(retrieved_titles, gold_titles):
    gold_set = set(gold_titles)

    if not gold_set:
        return 0.0

    matched = set(retrieved_titles) & gold_set
    return len(matched) / len(gold_set)

# Calculates the proportion of unique retrieved titles that match the expected titles
def title_precision_at_k(retrieved_titles, gold_titles):
    unique_retrieved = list(dict.fromkeys(retrieved_titles))

    if not unique_retrieved:
        return 0.0

    matched = set(unique_retrieved) & set(gold_titles)
    return len(matched) / len(unique_retrieved)

In [65]:
if standard_rag_test20:
    retrieval_evaluation = []

    for record, result in zip(beerqa_sample, standard_rag_test20):
        gold_titles = get_gold_titles(record)
        retrieved_titles = get_retrieved_titles(result, top_k=10)

        recall10 = title_recall_at_k(
            retrieved_titles,
            gold_titles,
        )

        precision10 = title_precision_at_k(
            retrieved_titles,
            gold_titles,
        )

        retrieval_evaluation.append({
            "id": record["id"],
            "question": record["question"],
            "gold_titles": gold_titles,
            "retrieved_titles": retrieved_titles,
            "title_recall@10": recall10,
            "title_precision@10": precision10,
        })

        result["gold_titles"] = gold_titles
        result["title_recall@10"] = recall10
        result["title_precision@10"] = precision10

    average_recall10 = sum(
        item["title_recall@10"]
        for item in retrieval_evaluation
    ) / len(retrieval_evaluation)

    average_precision10 = sum(
        item["title_precision@10"]
        for item in retrieval_evaluation
    ) / len(retrieval_evaluation)

    print("Questions evaluated:", len(retrieval_evaluation))
    print("Average Title Recall@10:", round(average_recall10, 4))
    print("Average Title Precision@10:", round(average_precision10, 4))

Questions evaluated: 20
Average Title Recall@10: 0.975
Average Title Precision@10: 0.605


In [66]:
if standard_rag_test20:
    with open(
        STANDARD_RAG_DEV20_EVAL_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        for result in standard_rag_test20:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

    print("Saved evaluated results:", STANDARD_RAG_DEV20_EVAL_PATH)

Saved evaluated results: C:\Users\Wei Khang\Documents\Master\FYP\processed_data\standard_rag_dev20_evaluated.jsonl
